# 70 · 대상 이용금액 문맥 복구와 순차 검색 개발 재검증

69의 source gate 실패는 보존한다. 이 실험은 원문 충돌을 본 후 고친 개발 평가이며 독립 확인/운영 승격이 아니다. 원래 두 운영 검색 프로필은 고정한다. 계획: notes/70_cap_context_repair_plan.md.

이 노트북이 단계별 실행의 기준이다. scripts/70_cap_context.py는 동일 규칙의 회귀검증,70_execution_runtime.py는 중단 재개·요청 결속·공통8달러 ledger 등록을 담당한다. HTML 생성 스크립트는 없다.


In [10]:
from pathlib import Path
import copy,importlib.util,json,os,sys,time,resource
from collections import Counter
from decimal import Decimal
import numpy as np
ROOT=Path.cwd()if(Path.cwd()/'notebooks').is_dir()else Path.cwd().parent
spec=importlib.util.spec_from_file_location('execution70_notebook',ROOT/'notebooks/scripts/70_execution_runtime.py')
H=importlib.util.module_from_spec(spec);sys.modules[spec.name]=H;spec.loader.exec_module(H)
V,T,R,A,F=H.V,H.T,H.R,H.A,H.F;BASE=H.BASE
os.chdir(ROOT)
SOURCE69=ROOT/'notebooks/data/69_repaired_sequential_confirmation'
NOTEBOOK=ROOT/'notebooks/70_cap_context_repair.ipynb'
ALLOW_NETWORK=os.environ.get('ALLOW_NETWORK','0')=='1'
MODES=('U','SEQ65','SEQ_REPAIRED');PROFILES=tuple(R.PROFILES)
def resources70(start,cpu,**extra):
 return dict(segment_wall_seconds=time.perf_counter()-start,segment_cpu_seconds=time.process_time()-cpu,
  process_lifetime_peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,ledger=T.budget_ledger(),
  first_token_seconds=None,browser_display_seconds=None,unmeasured=['TTFT','browser_display','actual_billed_invoice'],**extra)
print('70 런타임 등록, 기존68/69 불변, ALLOW_NETWORK=',ALLOW_NETWORK)


70 런타임 등록, 기존68/69 불변, ALLOW_NETWORK= False


## 68 영향 답변 재검토

최종 화면 60건을 재검토하고 기존432개를 같은66정책으로 재결속한다. 원형68 답변·지표는 수정하지 않는다.


In [1]:
# 68 원본은 읽기 전용. 알려진 오류 노출60건의 재검토 뒤 전432행을 같은 정책으로 재결속한다.
P68=ROOT/'notebooks/data/68_structured_evidence_generation'
reassess=BASE/'68_exposure_reassessment/reassessment.json'
audit=T.read(reassess);T.E.verify_file_map(audit['source_files_sha256'],ROOT,fresh=True)
original_reviews=T.read(P68/'manual_reviews.json')['records'];old_by={r['identity']:r for r in original_reviews}
old_jobs=T.read(P68/'jobs.json')['records'];old_job_by={j['identity']:j for j in old_jobs}
old_answers={i:T.read(P68/'answers'/(i+'.json'))for i in old_job_by}
left='H11:parent_child_bundle:C_old:r1';right='H11:parent_child_bundle:C_direct:r1'
left_reason=next(r['reason']for r in old_answers[left]['answer']['recommendations']if r['card_key']=='shinhan/Shinhan_Discount_Plan+_20250509')
right_reason=next(r['reason']for r in old_answers[right]['answer']['recommendations']if r['card_key']=='shinhan/Shinhan_Discount_Plan+_20250509')
assert '제공 근거에는 최소 결제금액이 명시되지 않았어요.'in left_reason
assert '최소 결제'not in right_reason
adjudication=dict(status='resolved_full_visible_surfaces_no_label_change',reviewed_by='main Codex',
 criterion='66 화면=본문+각recommendation.reason. 같은 문장을 반복할 필요는 없으며 한 화면의 근거로 다른 답변을 구제하지 않는다.',
 left=dict(identity=left,complete_DP_reason=left_reason,DP_minimum_not_stated_explained=True,old_D2='satisfied',retained_D2='satisfied'),
 right=dict(identity=right,complete_DP_reason=right_reason,DP_minimum_not_stated_explained=False,old_D2='missing',retained_D2='missing'),
 independent_candidate_false_positive='본문 비교에서는 차이가 안 보였지만 C_old의 실제 카드 추천이유에 최소결제 미기재 범위 설명이 있다.',
 cap_error_causal_claim=False,original_files_modified=False,
 source_files_sha256={str((P68/'answers'/(i+'.json')).relative_to(ROOT)):T.E.file_digest(P68/'answers'/(i+'.json'))for i in(left,right)})
T.write(BASE/'68_exposure_reassessment/main_adjudication.json',adjudication)
assert len(audit['records'])==60
for record in audit['records']:
 i=record['identity'];assert record['job_row_sha256']==R.P.sha(old_job_by[i])
 assert record['answer_row_sha256']==R.P.sha(old_answers[i])
 if record['technical_failure']:continue
 for req in record['requirements']:
  expected=next(x for x in old_by[i]['requirements']if x['id']==req['id'])['status']
  if i==left and req['id']=='H11.D2':assert req['status']=='uncertain'and expected=='satisfied'
  else:assert req['status']==expected
 assert record['visible_factual_accuracy'] in ('correct','uncertain')
I=H.module('integrity70_reassessment','68_review_integrity.py');E=I.E
context=copy.deepcopy(T.read(P68/'evaluation_context.json'))
context['followup_source_reassessment']={'path':str(reassess.relative_to(ROOT)),'cap_related_label_changes':0,'original68_unchanged':True}
for path in [reassess,BASE/'68_exposure_reassessment/validation_receipt.json',BASE/'68_exposure_reassessment/independent_visible_decisions.json',BASE/'68_exposure_reassessment/main_adjudication.json']:
 context['files_sha256'][str(path.relative_to(ROOT))]=T.E.file_digest(path)
refs=set(I.verify_union(T.read(P68/'source_union_manifest.json'),ROOT))
rubrics={r['query_id']:r for r in T.read(P68/'question_contract.json')['records']}
reviews=[];scores=[]
for index,job in enumerate(old_jobs,1):
 i=job['identity'];review=copy.deepcopy(old_by[i]);review.update(E.binding(job,old_answers[i],rubrics[job['query_id']],context))
 reviews.append(review);scores.append(I.score_review(job,old_answers[i],rubrics[job['query_id']],review,refs,context=context))
 if index%72==0:print('68 재결속 채점',index,'/432',flush=True)
summary=E.aggregate(scores,query_ids=list(rubrics),profiles=sorted({j['profile']for j in old_jobs}),modes=['A_old','A_direct','B_old','B_direct','C_old','C_direct'],repeats=[1,2,3],context=context)
assert summary==T.read(P68/'evaluation_summary.json')['conditions']
# 원래 선택 셀을 원형 그대로 재실행하되 모든 쓰기를 capture하여 원본68을 보호한다.
selection_source=''.join(next(c['source']for c in T.read(ROOT/'notebooks/68_structured_evidence_generation.ipynb')['cells']if c['id']=='exp68-selection-01'))
assert E.digest(selection_source)==T.read(P68/'selection.json')['selection_cell_source_sha256']
captured={};old_frozen=R.X.frozen
try:
 R.X.frozen=lambda path,value:captured.setdefault(str(path),copy.deepcopy(value))
 scope=dict(ROOT=ROOT,BASE=P68,T=T,jobs=old_jobs,rubric_by_id=rubrics,json=json)
 exec(compile(selection_source,'original68_selection_replay','exec'),scope)
finally:R.X.frozen=old_frozen
assert captured[str(P68/'selection.json')]==T.read(P68/'selection.json')
assert captured[str(P68/'selection_analysis.json')]==T.read(P68/'selection_analysis.json')
folder=BASE/'68_reassessment_full'
T.write(folder/'evaluation_context.json',context)
T.write(folder/'manual_reviews.json',dict(records=reviews,evaluation_context_sha256=E.digest(context),semantics='Same fixed labels reaffirmed after60 exposure reviews; new bindings only'))
T.write(folder/'evaluation_results.json',dict(records=scores,evaluation_context_sha256=E.digest(context)))
T.write(folder/'evaluation_summary.json',dict(conditions=summary,logical_answers=432,semantic_label_changes=0,old_68_unchanged=True))
selection=copy.deepcopy(T.read(P68/'selection.json'))
selection.update(status='reaffirmed68_development_settings_after_cap_exposure_reassessment',records=captured[str(P68/'selection.json')]['records'],original_selection_raw_sha256=T.E.file_digest(P68/'selection.json'),
 new_responses=0,semantic_label_changes=0,full_matrix_rebound_and_aggregated=432,
 original_selection_code_replayed_without_writes=True,
 files_sha256={str(p.relative_to(ROOT)):T.E.file_digest(p)for p in [reassess,BASE/'68_exposure_reassessment/main_adjudication.json',folder/'evaluation_context.json',folder/'manual_reviews.json',folder/'evaluation_results.json',folder/'evaluation_summary.json']})
T.write(BASE/'reaffirmed68_selection.json',selection)
print('68 선택 재확인: benefit B_direct, parent B_old. cap 오류 노출에 따른 최종 판정변경0. 원68 불변/API0.')


68 재결속 채점 432 /432
68 선택 재확인: benefit B_direct, parent B_old. cap 오류 노출에 따른 최종 판정변경0. 원68 불변/API0.


## 원문 소유자 검증과 사본 DB5 재현

등록 database-manager가 실제 적용한 동일 코드를 재실행한다. 이미 있으면 정확히 일치할 때만 재사용한다. 원DB4/raw/PDF는 수정하지 않는다. SQL의 cap_raw는 DB5에서만 문맥 보완 필드이며 raw_json은 원형 보존용으로 분리된다.


In [2]:
from __future__ import annotations

import copy
import hashlib
import importlib.util
import json
import shutil
import sqlite3
import tempfile
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().resolve()
SOURCE_DIR = ROOT / "notebooks/data/67_structured_retrieval_repair/db_v4"
OUTPUT_ROOT = ROOT / "notebooks/data/70_cap_context_repair"
OUTPUT_DB_DIR = OUTPUT_ROOT / "db_v5"
CHANGES_PATH = OUTPUT_ROOT / "db_changes.json"
RECEIPT_PATH = OUTPUT_ROOT / "db_migration_receipt.json"
PREVIEW_PATH = OUTPUT_ROOT / "cap_patch_preview.json"
HELPER_PATH = ROOT / "notebooks/scripts/70_cap_context.py"
DATABASE_VERSION = "structured-benefit-experiment-db5-cap-context70-v1"
EXPECTED_SOURCE_HASHES = {
    "structured.sqlite": "76387931e46e85686cf64ffd656c27929cd955ed922b53f201fd61901fc01c8d",
    "documents.jsonl": "fdf855a8d3b11103f776f913b010860c6b9c5e53af5b63a61738683c3d678eac",
    "facts.jsonl": "9cf49d69fda363e495f050401b2b915973a4fac5c4084110fa04cdc1022b34c8",
}
EXPECTED_HELPER_SHA256 = "5987e53f8ee67c8dc0d12ed7427472efa906e7aacbd9722b63a1d9fc4bba31c2"
EXPECTED_PREVIEW_SHA256 = "b820002f20f8c728ac18ef27903f709941f19c048442dcc7593f9f67b4d2a5ec"
ALLOWED_DERIVED_FIELDS = {
    "cap_krw",
    "cap_state",
    "cap_reason",
    "cap_period",
    "cap_period_state",
    "cap_period_reason",
}
ALLOWED_SQL_FACT_COLUMNS = {
    "cap_raw",
    "cap_krw",
    "cap_state",
    "cap_period",
    "derived_json",
}


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]


def jsonl_bytes(rows: list[dict]) -> bytes:
    return (
        "".join(
            json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n"
            for row in rows
        )
    ).encode("utf-8")


def json_bytes(value: object) -> bytes:
    return (
        json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + "\n"
    ).encode("utf-8")


def load_helper():
    spec = importlib.util.spec_from_file_location("cap_context70_migration", HELPER_PATH)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def sqlite_schema(connection: sqlite3.Connection) -> list[tuple]:
    return connection.execute(
        "SELECT type,name,tbl_name,sql FROM sqlite_master "
        "WHERE name NOT LIKE 'sqlite_%' ORDER BY type,name"
    ).fetchall()


def table_rows(
    connection: sqlite3.Connection, table: str, order_by: str
) -> tuple[list[str], list[tuple]]:
    columns = [
        row[1] for row in connection.execute(f'PRAGMA table_info("{table}")')
    ]
    projection = ",".join(f'"{column}"' for column in columns)
    rows = connection.execute(
        f'SELECT {projection} FROM "{table}" ORDER BY {order_by}'
    ).fetchall()
    return columns, rows


prepared_receipt = json.loads(RECEIPT_PATH.read_text(encoding="utf-8"))
execution_code = prepared_receipt["execution_code"]
migration_code_sha256 = sha256_bytes(execution_code.encode("utf-8"))

source_hashes_before = {
    name: sha256_file(SOURCE_DIR / name) for name in EXPECTED_SOURCE_HASHES
}
if source_hashes_before != EXPECTED_SOURCE_HASHES:
    raise RuntimeError(f"DB4 source hash mismatch: {source_hashes_before}")
if sha256_file(HELPER_PATH) != EXPECTED_HELPER_SHA256:
    raise RuntimeError("70_cap_context.py hash mismatch")
if sha256_file(PREVIEW_PATH) != EXPECTED_PREVIEW_SHA256:
    raise RuntimeError("cap_patch_preview.json hash mismatch")

helper = load_helper()
documents = read_jsonl(SOURCE_DIR / "documents.jsonl")
facts = read_jsonl(SOURCE_DIR / "facts.jsonl")
preview = json.loads(PREVIEW_PATH.read_text(encoding="utf-8"))
if len(documents) != 106 or len(facts) != 3403:
    raise RuntimeError("unexpected DB4 corpus size")
if preview.get("facts_scanned") != 3403 or len(preview.get("records", [])) != 3:
    raise RuntimeError("unexpected preview size")
if len({doc["document_id"] for doc in documents}) != 106:
    raise RuntimeError("duplicate document_id")
if len({fact["fact_id"] for fact in facts}) != 3403:
    raise RuntimeError("duplicate fact_id")

documents_by_id = {doc["document_id"]: doc for doc in documents}
fact_counts = Counter(fact["document_id"] for fact in facts)
normalized_sha_errors = []
pdf_sha_errors = []
document_fact_count_errors = []
for doc in documents:
    document_id = doc["document_id"]
    normalized_path = ROOT / doc["source_relative_path"]
    pdf_path = ROOT / "data/raw" / f"{document_id}.pdf"
    if not normalized_path.is_file() or sha256_file(normalized_path) != doc["normalized_json_sha256"]:
        normalized_sha_errors.append(document_id)
    if not pdf_path.is_file() or sha256_file(pdf_path) != doc["source_pdf_sha256"]:
        pdf_sha_errors.append(document_id)
    if fact_counts[document_id] != doc["fact_count"]:
        document_fact_count_errors.append(document_id)
if normalized_sha_errors or pdf_sha_errors or document_fact_count_errors:
    raise RuntimeError(
        "document provenance validation failed: "
        f"normalized={normalized_sha_errors}, pdf={pdf_sha_errors}, "
        f"counts={document_fact_count_errors}"
    )

fact_provenance_errors = []
for fact in facts:
    doc = documents_by_id.get(fact["document_id"])
    if (
        doc is None
        or fact["source_relative_path"] != doc["source_relative_path"]
        or fact["source_path_sha256"] != doc["source_path_sha256"]
        or fact["source_pdf_sha256"] != doc["source_pdf_sha256"]
    ):
        fact_provenance_errors.append(fact["fact_id"])
if fact_provenance_errors:
    raise RuntimeError(f"fact provenance mismatch: {fact_provenance_errors}")

source_db_uri = f"file:{SOURCE_DIR / 'structured.sqlite'}?mode=ro&immutable=1"
source_connection = sqlite3.connect(source_db_uri, uri=True)
try:
    if source_connection.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
        raise RuntimeError("DB4 integrity failure")
    if source_connection.execute("PRAGMA foreign_key_check").fetchall():
        raise RuntimeError("DB4 foreign-key failure")
    source_schema = sqlite_schema(source_connection)
    source_fact_columns, source_fact_rows = table_rows(
        source_connection, "facts", "document_id,ordinal"
    )
    source_document_columns, source_document_rows = table_rows(
        source_connection, "documents", "document_id"
    )
    source_alias_columns, source_alias_rows = table_rows(
        source_connection, "target_aliases", "alias_group,alias"
    )
    source_metadata = {
        key: json.loads(value)
        for key, value in source_connection.execute(
            "SELECT key,value_json FROM metadata ORDER BY key"
        )
    }
finally:
    source_connection.close()

if len(source_fact_rows) != 3403 or len(source_document_rows) != 106 or len(source_alias_rows) != 22:
    raise RuntimeError("DB4 SQL row count mismatch")
source_fact_index = {
    row[source_fact_columns.index("fact_id")]: row for row in source_fact_rows
}
source_document_index = {
    row[source_document_columns.index("document_id")]: row
    for row in source_document_rows
}
for doc in documents:
    sql_row = source_document_index[doc["document_id"]]
    if json.loads(sql_row[source_document_columns.index("raw_json")]) != doc:
        raise RuntimeError(f"document JSONL/SQLite mismatch: {doc['document_id']}")
for fact in facts:
    sql_row = source_fact_index[fact["fact_id"]]
    if json.loads(sql_row[source_fact_columns.index("raw_json")]) != fact["raw_fact"]:
        raise RuntimeError(f"raw_fact/SQLite raw_json mismatch: {fact['fact_id']}")
    if json.loads(sql_row[source_fact_columns.index("derived_json")]) != fact["derived"]:
        raise RuntimeError(f"derived/SQLite mismatch: {fact['fact_id']}")
    if sql_row[source_fact_columns.index("cap_raw")] != fact["fields"]["cap"]:
        raise RuntimeError(f"cap field/SQLite mismatch: {fact['fact_id']}")

line_maps = {
    document_id: helper.source_lines(doc)
    for document_id, doc in documents_by_id.items()
}
computed_patches = {}
for fact in facts:
    patch = helper.cap_context_patch(fact, line_maps[fact["document_id"]])
    if patch is not None:
        computed_patches[fact["fact_id"]] = patch
preview_by_id = {record["fact_id"]: record for record in preview["records"]}
if set(computed_patches) != set(preview_by_id):
    raise RuntimeError(
        f"preview ID mismatch: computed={sorted(computed_patches)}, "
        f"preview={sorted(preview_by_id)}"
    )
for fact_id, record in preview_by_id.items():
    fact = next(item for item in facts if item["fact_id"] == fact_id)
    if (
        record["owner"] != fact["document_id"]
        or record["ordinal"] != fact["ordinal"]
        or record["old_cap"] != fact["fields"]["cap"]
        or record["patch"] != computed_patches[fact_id]
        or any(
            record["old_derived"].get(key) != fact["derived"].get(key)
            for key in ALLOWED_DERIVED_FIELDS
        )
    ):
        raise RuntimeError(f"preview content mismatch: {fact_id}")

new_facts = []
changes = []
for fact in facts:
    fact_id = fact["fact_id"]
    if fact_id not in computed_patches:
        new_facts.append(copy.deepcopy(fact))
        continue
    patch = computed_patches[fact_id]
    updated = copy.deepcopy(fact)
    updated["fields"] = patch["fields"]
    updated["field_evidence"] = patch["field_evidence"]
    updated["derived"] = patch["derived"]

    if {
        key: value for key, value in updated.items()
        if key not in {"fields", "field_evidence", "derived"}
    } != {
        key: value for key, value in fact.items()
        if key not in {"fields", "field_evidence", "derived"}
    }:
        raise RuntimeError(f"forbidden top-level fact change: {fact_id}")
    if {
        key: value for key, value in updated["fields"].items() if key != "cap"
    } != {
        key: value for key, value in fact["fields"].items() if key != "cap"
    }:
        raise RuntimeError(f"forbidden normalized field change: {fact_id}")
    if {
        key: value for key, value in updated["field_evidence"].items()
        if key != "cap"
    } != {
        key: value for key, value in fact["field_evidence"].items()
        if key != "cap"
    }:
        raise RuntimeError(f"forbidden evidence change: {fact_id}")
    changed_derived_keys = {
        key
        for key in set(fact["derived"]) | set(updated["derived"])
        if fact["derived"].get(key) != updated["derived"].get(key)
    }
    if not changed_derived_keys <= ALLOWED_DERIVED_FIELDS:
        raise RuntimeError(
            f"forbidden derived change: {fact_id} {sorted(changed_derived_keys)}"
        )
    if updated["raw_fact"] != fact["raw_fact"]:
        raise RuntimeError(f"raw_fact changed: {fact_id}")

    changes.append(
        {
            "fact_id": fact_id,
            "document_id": fact["document_id"],
            "ordinal": fact["ordinal"],
            "rule": patch["rule"],
            "source_line_id": patch["source_line_id"],
            "source_line": patch["source_line"],
            "qualifier_evidence": patch["qualifier_evidence"],
            "original_cap_evidence": patch["original_cap_evidence"],
            "changes": {
                "fields.cap": {
                    "before": fact["fields"]["cap"],
                    "after": updated["fields"]["cap"],
                },
                "field_evidence.cap": {
                    "before": fact["field_evidence"]["cap"],
                    "after": updated["field_evidence"]["cap"],
                },
                **{
                    f"derived.{key}": {
                        "before": fact["derived"].get(key),
                        "after": updated["derived"].get(key),
                    }
                    for key in sorted(ALLOWED_DERIVED_FIELDS)
                    if fact["derived"].get(key) != updated["derived"].get(key)
                },
            },
        }
    )
    new_facts.append(updated)

if len(changes) != 3 or {item["fact_id"] for item in changes} != set(preview_by_id):
    raise RuntimeError("migration change set is not the exact preview three")

idempotence_hits = [
    fact["fact_id"]
    for fact in new_facts
    if helper.cap_context_patch(
        fact, line_maps[fact["document_id"]]
    ) is not None
]
if idempotence_hits:
    raise RuntimeError(f"migration is not idempotent: {idempotence_hits}")

changes_payload = {
    "version": DATABASE_VERSION,
    "rule": helper.VERSION,
    "parent_files_sha256": source_hashes_before,
    "helper_sha256": EXPECTED_HELPER_SHA256,
    "preview_sha256": EXPECTED_PREVIEW_SHA256,
    "changed_fact_count": 3,
    "records": changes,
    "scope": (
        "Only normalized fields.cap, field_evidence.cap, and six derived cap "
        "fields change. raw_fact and SQLite raw_json remain the immutable "
        "source extraction."
    ),
}
changes_bytes = json_bytes(changes_payload)
facts_bytes = jsonl_bytes(new_facts)
documents_bytes = (SOURCE_DIR / "documents.jsonl").read_bytes()

with tempfile.TemporaryDirectory(prefix="pickcardu70_") as temporary:
    temporary_root = Path(temporary)
    planned_db = temporary_root / "structured.sqlite"
    shutil.copyfile(SOURCE_DIR / "structured.sqlite", planned_db)
    connection = sqlite3.connect(planned_db)
    try:
        connection.execute("PRAGMA foreign_keys=ON")
        connection.execute("BEGIN IMMEDIATE")
        for updated in new_facts:
            if updated["fact_id"] not in preview_by_id:
                continue
            derived = updated["derived"]
            cursor = connection.execute(
                "UPDATE facts SET cap_raw=?,cap_krw=?,cap_state=?,cap_period=?,"
                "derived_json=? WHERE fact_id=?",
                (
                    updated["fields"]["cap"],
                    derived.get("cap_krw"),
                    derived["cap_state"],
                    derived.get("cap_period"),
                    json.dumps(derived, ensure_ascii=False, sort_keys=True),
                    updated["fact_id"],
                ),
            )
            if cursor.rowcount != 1:
                raise RuntimeError(f"SQL update count mismatch: {updated['fact_id']}")

        counts = copy.deepcopy(source_metadata["counts"])
        counts["known_cap"] = connection.execute(
            "SELECT count(*) FROM facts WHERE cap_state='known'"
        ).fetchone()[0]
        support_rates = copy.deepcopy(source_metadata["support_rates"])
        support_rates["known_cap"] = counts["known_cap"] / counts["facts"]
        normalization70 = {
            "version": helper.VERSION,
            "database_version": DATABASE_VERSION,
            "parent": {
                "path": "notebooks/data/67_structured_retrieval_repair/db_v4",
                "files_sha256": source_hashes_before,
            },
            "applied_fact_count": 3,
            "applied_fact_ids": sorted(preview_by_id),
            "preview_sha256": EXPECTED_PREVIEW_SHA256,
            "rule_code_sha256": EXPECTED_HELPER_SHA256,
            "migration_code_sha256": migration_code_sha256,
            "raw_normalized_divergence": (
                "SQLite raw_json and JSONL raw_fact preserve the original "
                "extraction. DB5 cap_raw and JSONL fields.cap carry the "
                "experiment-only normalized cap context."
            ),
        }
        metadata_updates = {
            "schema_version": DATABASE_VERSION,
            "counts": counts,
            "support_rates": support_rates,
            "normalization70": normalization70,
        }
        for key, value in metadata_updates.items():
            connection.execute(
                "INSERT INTO metadata(key,value_json) VALUES (?,?) "
                "ON CONFLICT(key) DO UPDATE SET value_json=excluded.value_json",
                (key, json.dumps(value, ensure_ascii=False, sort_keys=True)),
            )
        if connection.execute("PRAGMA foreign_key_check").fetchall():
            raise RuntimeError("DB5 foreign-key failure before commit")
        if connection.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
            raise RuntimeError("DB5 integrity failure before commit")
        connection.commit()
    except Exception:
        connection.rollback()
        raise
    finally:
        connection.close()

    output_connection = sqlite3.connect(
        f"file:{planned_db}?mode=ro&immutable=1", uri=True
    )
    try:
        integrity = output_connection.execute(
            "PRAGMA integrity_check"
        ).fetchone()[0]
        foreign_keys = output_connection.execute(
            "PRAGMA foreign_key_check"
        ).fetchall()
        output_schema = sqlite_schema(output_connection)
        output_fact_columns, output_fact_rows = table_rows(
            output_connection, "facts", "document_id,ordinal"
        )
        output_document_columns, output_document_rows = table_rows(
            output_connection, "documents", "document_id"
        )
        output_alias_columns, output_alias_rows = table_rows(
            output_connection, "target_aliases", "alias_group,alias"
        )
        output_metadata = {
            key: json.loads(value)
            for key, value in output_connection.execute(
                "SELECT key,value_json FROM metadata ORDER BY key"
            )
        }
    finally:
        output_connection.close()

    if integrity != "ok" or foreign_keys:
        raise RuntimeError("DB5 post-write integrity failure")
    if output_schema != source_schema:
        raise RuntimeError("SQLite schema changed")
    if output_fact_columns != source_fact_columns:
        raise RuntimeError("facts columns changed")
    if output_document_columns != source_document_columns:
        raise RuntimeError("documents columns changed")
    if output_alias_columns != source_alias_columns:
        raise RuntimeError("target_alias columns changed")
    if output_document_rows != source_document_rows:
        raise RuntimeError("document SQL rows changed")
    if output_alias_rows != source_alias_rows:
        raise RuntimeError("target alias rows changed")

    output_fact_index = {
        row[output_fact_columns.index("fact_id")]: row for row in output_fact_rows
    }
    changed_sql_ids = []
    unchanged_fact_rows = 0
    for fact_id, source_row in source_fact_index.items():
        output_row = output_fact_index[fact_id]
        differing_columns = {
            column
            for column, before, after in zip(
                source_fact_columns, source_row, output_row
            )
            if before != after
        }
        if differing_columns:
            changed_sql_ids.append(fact_id)
            if not differing_columns <= ALLOWED_SQL_FACT_COLUMNS:
                raise RuntimeError(
                    f"forbidden SQL fact columns changed: "
                    f"{fact_id} {sorted(differing_columns)}"
                )
        else:
            unchanged_fact_rows += 1
    if set(changed_sql_ids) != set(preview_by_id) or unchanged_fact_rows != 3400:
        raise RuntimeError(
            f"SQL change set mismatch: changed={changed_sql_ids}, "
            f"unchanged={unchanged_fact_rows}"
        )

    allowed_metadata_keys = {
        "schema_version",
        "counts",
        "support_rates",
        "normalization70",
    }
    for key, value in source_metadata.items():
        if key not in allowed_metadata_keys and output_metadata.get(key) != value:
            raise RuntimeError(f"unapproved metadata change: {key}")

    new_facts_by_id = {fact["fact_id"]: fact for fact in new_facts}
    sql_json_mismatches = []
    raw_json_mismatches = []
    cap_scalar_mismatches = []
    for fact_id, output_row in output_fact_index.items():
        fact = new_facts_by_id[fact_id]
        raw_json = json.loads(output_row[output_fact_columns.index("raw_json")])
        derived_json = json.loads(
            output_row[output_fact_columns.index("derived_json")]
        )
        if raw_json != fact["raw_fact"]:
            raw_json_mismatches.append(fact_id)
        if (
            output_row[output_fact_columns.index("cap_raw")]
            != fact["fields"]["cap"]
            or derived_json != fact["derived"]
        ):
            sql_json_mismatches.append(fact_id)
        if (
            output_row[output_fact_columns.index("cap_krw")]
            != fact["derived"].get("cap_krw")
            or output_row[output_fact_columns.index("cap_state")]
            != fact["derived"]["cap_state"]
            or output_row[output_fact_columns.index("cap_period")]
            != fact["derived"].get("cap_period")
        ):
            cap_scalar_mismatches.append(fact_id)
    if raw_json_mismatches or sql_json_mismatches or cap_scalar_mismatches:
        raise RuntimeError(
            "SQLite/JSONL validation failed: "
            f"raw={raw_json_mismatches}, normalized={sql_json_mismatches}, "
            f"scalar={cap_scalar_mismatches}"
        )

    source_hashes_after = {
        name: sha256_file(SOURCE_DIR / name)
        for name in EXPECTED_SOURCE_HASHES
    }
    if source_hashes_after != source_hashes_before:
        raise RuntimeError("DB4 source files changed during migration")

    planned_db_bytes = planned_db.read_bytes()
    planned_outputs = {
        OUTPUT_DB_DIR / "structured.sqlite": planned_db_bytes,
        OUTPUT_DB_DIR / "documents.jsonl": documents_bytes,
        OUTPUT_DB_DIR / "facts.jsonl": facts_bytes,
        CHANGES_PATH: changes_bytes,
    }
    output_hashes = {
        path.relative_to(OUTPUT_ROOT).as_posix(): sha256_bytes(content)
        for path, content in planned_outputs.items()
    }
    receipt = {
        "status": "applied_and_validated",
        "environment": "local experiment copy",
        "database_version": DATABASE_VERSION,
        "parent": {
            "path": "notebooks/data/67_structured_retrieval_repair/db_v4",
            "files_sha256_before": source_hashes_before,
            "files_sha256_after": source_hashes_after,
        },
        "rule": {
            "name": helper.VERSION,
            "code_path": "notebooks/scripts/70_cap_context.py",
            "code_sha256": EXPECTED_HELPER_SHA256,
            "preview_path": (
                "notebooks/data/70_cap_context_repair/"
                "cap_patch_preview.json"
            ),
            "preview_sha256": EXPECTED_PREVIEW_SHA256,
        },
        "execution_code": execution_code,
        "execution_code_sha256": migration_code_sha256,
        "execution_command": (
            "PYTHONDONTWRITEBYTECODE=1 "
            "/home/sms/miniconda3/envs/skn25/bin/python -B -c "
            "\"import json; from pathlib import Path; "
            "p=Path('notebooks/data/70_cap_context_repair/"
            "db_migration_receipt.json'); "
            "exec(compile(json.loads(p.read_text(encoding='utf-8'))"
            "['execution_code'],str(p),'exec'))\""
        ),
        "changes_path": (
            "notebooks/data/70_cap_context_repair/db_changes.json"
        ),
        "changed_fact_count": 3,
        "changed_fact_ids": sorted(preview_by_id),
        "output_sha256": output_hashes,
        "validation": {
            "documents": len(documents),
            "facts": len(facts),
            "aliases": len(output_alias_rows),
            "pages": sum(doc["page_count"] for doc in documents),
            "normalized_json_sha_match_count": len(documents),
            "pdf_sha_match_count": len(documents),
            "document_fact_ownership_match_count": len(documents),
            "preview_exact_match": True,
            "sqlite_integrity_check": integrity,
            "foreign_key_error_count": len(foreign_keys),
            "schema_identical": output_schema == source_schema,
            "documents_sql_identical": output_document_rows == source_document_rows,
            "documents_jsonl_byte_identical": documents_bytes
            == (SOURCE_DIR / "documents.jsonl").read_bytes(),
            "target_aliases_identical": output_alias_rows == source_alias_rows,
            "unchanged_fact_row_count": unchanged_fact_rows,
            "changed_sql_fact_ids_exact": sorted(changed_sql_ids)
            == sorted(preview_by_id),
            "raw_json_preserved_all_facts": not raw_json_mismatches,
            "sqlite_jsonl_normalized_cap_identical": not sql_json_mismatches,
            "sqlite_cap_scalar_derived_identical": not cap_scalar_mismatches,
            "idempotence_remaining_patch_count": len(idempotence_hits),
            "source_files_unchanged": source_hashes_after
            == source_hashes_before,
        },
        "raw_normalized_divergence": (
            "This is an experiment-only normalized copy: SQLite raw_json and "
            "JSONL raw_fact retain the source extraction, while DB5 cap_raw, "
            "JSONL fields.cap, field_evidence.cap, and six derived cap fields "
            "contain the rule70 context repair."
        ),
        "source_audit_semantic_promotion": False,
        "answer_gate_changed": False,
    }
    receipt_bytes = json_bytes(receipt)

    expected_db_files = {
        Path("structured.sqlite"),
        Path("documents.jsonl"),
        Path("facts.jsonl"),
    }
    if OUTPUT_DB_DIR.exists():
        actual_db_files = {
            path.relative_to(OUTPUT_DB_DIR)
            for path in OUTPUT_DB_DIR.rglob("*")
            if path.is_file()
        }
        if actual_db_files != expected_db_files:
            raise RuntimeError(
                f"existing db_v5 file set conflicts: {sorted(actual_db_files)}"
            )
        for path, content in planned_outputs.items():
            if path.parent == OUTPUT_DB_DIR and path.read_bytes() != content:
                raise RuntimeError(f"existing output conflicts: {path}")
    else:
        OUTPUT_DB_DIR.mkdir(parents=True)
        for path, content in planned_outputs.items():
            if path.parent == OUTPUT_DB_DIR:
                path.write_bytes(content)

    if CHANGES_PATH.exists():
        if CHANGES_PATH.read_bytes() != changes_bytes:
            raise RuntimeError("existing db_changes.json conflicts")
    else:
        CHANGES_PATH.write_bytes(changes_bytes)

    if RECEIPT_PATH.exists():
        existing_receipt = json.loads(RECEIPT_PATH.read_text(encoding="utf-8"))
        if existing_receipt.get("status") == "prepared_execution_code":
            RECEIPT_PATH.write_bytes(receipt_bytes)
        elif RECEIPT_PATH.read_bytes() != receipt_bytes:
            raise RuntimeError("existing db_migration_receipt.json conflicts")
    else:
        raise RuntimeError("execution receipt disappeared during migration")

print(
    json.dumps(
        {
            "status": "applied_and_validated",
            "changed_fact_count": 3,
            "changed_fact_ids": sorted(preview_by_id),
            "output_sha256": output_hashes,
            "sqlite_integrity_check": integrity,
            "foreign_key_error_count": len(foreign_keys),
            "unchanged_fact_row_count": unchanged_fact_rows,
            "idempotence_remaining_patch_count": len(idempotence_hits),
        },
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
)

{
  "changed_fact_count": 3,
  "changed_fact_ids": [
    "fact_3a7a67111deecc01a2d8c659",
    "fact_6547be3b20be0ac90a6f3771",
    "fact_7923e642f3e8c43f426fade3"
  ],
  "foreign_key_error_count": 0,
  "idempotence_remaining_patch_count": 0,
  "output_sha256": {
    "db_changes.json": "7094168c7d716d2648c92fb9a253c6a4bfd1b7109ff7078d27ae6b773021fd9e",
    "db_v5/documents.jsonl": "fdf855a8d3b11103f776f913b010860c6b9c5e53af5b63a61738683c3d678eac",
    "db_v5/facts.jsonl": "de1b4e5b2b420b14bd274e8c6a4386525ec6602f221c76e6b3fe9c299f2ff853",
    "db_v5/structured.sqlite": "5b6c734c21fd06bd55f381a2c71d631731b26d72751854ec0890b8e5eca6964c"
  },
  "sqlite_integrity_check": "ok",
  "status": "applied_and_validated",
  "unchanged_fact_row_count": 3400
}


## 설정 동결·69 추출/벡터 원응답 결속

네트워크 호출 없이 검증·조립한다. 원질문/운영 프로필/기존 조건 추출 결과는 고정한다.


In [3]:
# 실제 호출 전 모든 실행 셀과 설정을 동결한다. 질문/추출/임베딩은69원자료에 결속해 재사용한다.
start=time.perf_counter();cpu=time.process_time()
selected=T.read(BASE/'reaffirmed68_selection.json');T.E.verify_file_map(selected['files_sha256'],ROOT,fresh=True)
assert selected['semantic_label_changes']==0 and selected['production_promotion_allowed']is False
profiles={r['profile']:{k:r[k]for k in('input_arm','prompt_variant','mode')}for r in selected['records']}
assert profiles==T.read(SOURCE69/'configuration_contract.json')['profiles']
migration=T.read(BASE/'db_migration_receipt.json')
assert migration['changed_fact_count']==3 and migration['source_audit_semantic_promotion']is False
DB5=BASE/'db_v5';DB3=ROOT/'notebooks/data/60_structured_retrieval_repair/db_v3'
for name in('questions.json','question_contract.json','parser_results.json','source_registry.json','source_proof_inventory.json'):
 source=SOURCE69/name;target=BASE/name
 if target.exists():assert target.read_bytes()==source.read_bytes()
 else:target.write_bytes(source.read_bytes())
queries=T.read(BASE/'questions.json')['queries'];rubric=T.read(BASE/'question_contract.json')
parsers={(r['query_id'],r['repeat']):r for r in T.read(BASE/'parser_results.json')['records']}
assert len(queries)==24 and len(parsers)==72
parser_manifest=T.read(SOURCE69/'parser_old/execution_manifest.json');T.guard_manifest(parser_manifest)
for q in queries:
 for repeat in(1,2,3):T.validate_parser_checkpoint(parsers[q['query_id'],repeat],q,parser_manifest,SOURCE69/'parser_old')
# 원69의 exact 요청 raw와 벡터를 확인한다. raw를70 api_cache로 복사해 중복 회계하지 않는다.
request=dict(model='text-embedding-3-small',input=[q['question']for q in queries],dimensions=1536)
raw_path=SOURCE69/'api_cache'/('embedding_batch_'+R.P.sha(request)+'.json')
raw=T.read(raw_path);ec=T.read(SOURCE69/'embedding_contract.json')
assert raw['status']=='ok'and raw['request']==request and raw['usage']==raw['raw_response']['usage']
assert ec['request_sha256']==R.P.sha(request)and ec['raw_file_sha256']==T.E.file_digest(raw_path)and ec['questions_sha256']==R.P.sha(queries)
ordered=sorted(raw['raw_response']['data'],key=lambda x:x['index']);assert [x['index']for x in ordered]==list(range(24))
vectors=np.load(SOURCE69/'query_vectors.npy',allow_pickle=False)
assert vectors.shape==(24,1536)and np.isfinite(vectors).all()and np.array_equal(vectors,np.asarray([x['embedding']for x in ordered],dtype=np.float32))
assert ec['matrix_sha256']==R.P.sha(vectors.tobytes())
if (BASE/'query_vectors.npy').exists():assert np.array_equal(np.load(BASE/'query_vectors.npy',allow_pickle=False),vectors)
else:np.save(BASE/'query_vectors.npy',vectors)
reuse=dict(status='exact69_parser_and_embedding_reuse_no_new_provider_requests',parser_rows=72,query_vectors=24,
 sources_sha256={str(p.relative_to(ROOT)):T.E.file_digest(p)for p in [SOURCE69/'parser_results.json',SOURCE69/'parser_old/execution_manifest.json',raw_path,SOURCE69/'embedding_contract.json',SOURCE69/'query_vectors.npy']},
 raw_embedding_not_copied_into70_api_cache=True)
R.X.frozen(BASE/'parser_embedding_reuse.json',reuse)
old_manifest=T.read(SOURCE69/'execution_manifest.json');T.guard_manifest(old_manifest)
config=dict(status='posthoc_source_repair_development_not_independent_confirmation',profiles=profiles,modes=list(MODES),
 DB5_path=str(DB5.relative_to(ROOT)),parent_DB4='notebooks/data/67_structured_retrieval_repair/db_v4',matcher='projected',parser='reused_old69',
 U='original operating unstructured, old prompt',SEQ65='DB3 baseline matcher, original U, old prompt',
 SEQ_REPAIRED='DB5 projected matcher, original U prefix plus selected68 input/prompt',
 original_U_union=False,profile_retuning=False,logical_answers=432,maximum_answer_requests=864,new_parser_requests=0,new_embedding_requests=0,
 production_promotion=False,question_block_bootstrap=dict(iterations=10000,seed=70,confidence=.95),
 known_U_source_conflict='Monthlyreward source150만원 vs OCR50만원 remains; do not claim all inputs source-correct')
R.X.frozen(BASE/'configuration_contract.json',config)
sources=copy.deepcopy(old_manifest['source_sha256'])
for name in('70_execution_runtime.py','70_cap_context.py'):
 sources['notebooks/scripts/'+name]=T.E.file_digest(ROOT/'notebooks/scripts'/name)
critical=copy.deepcopy(old_manifest['critical_data_sha256'])
critical.update(selected['files_sha256'])
for folder,names in [(BASE,['configuration_contract.json','questions.json','question_contract.json','parser_results.json','query_vectors.npy','parser_embedding_reuse.json','reaffirmed68_selection.json','db_migration_receipt.json','db_changes.json']),
                     (DB5,['structured.sqlite','facts.jsonl','documents.jsonl']),
                     (SOURCE69,['execution_manifest.json','retrieval_results.json','source_inventory.json'])]:
 for name in names:critical[str((folder/name).relative_to(ROOT))]=T.E.file_digest(folder/name)
critical.update(reuse['sources_sha256'])
# DB5 출처와 기존 row seal을 동결한다. 실제 원문 의미 확인은 source gate에서 따로 한다.
for row in T.read(SOURCE69/'retrieval_results.json')['records']:
 path=SOURCE69/'retrievals'/(row['identity']+'.seal.json');critical[str(path.relative_to(ROOT))]=T.E.file_digest(path)
identities=[f"{q['query_id']}:{p}:{m}:r{n}"for q in queries for p in PROFILES for m in MODES for n in(1,2,3)]
variants={i:profiles[i.split(':')[1]]['prompt_variant']if i.split(':')[2]=='SEQ_REPAIRED'else'old'for i in identities}
prompts=R.G.C.operating_prompts()
manifest=dict(stage=70,model='gpt-6-luna',reasoning='medium',source_sha256=sources,critical_data_sha256=critical,
 notebook_phase_code_sha256={str(NOTEBOOK.relative_to(ROOT)):dict(prefix='exp70-exec-',sha256=T.phase_code_sha(NOTEBOOK,'exp70-exec-'))},
 notebook_code_sha256={},data_sha256=old_manifest['data_sha256'],allowed_identities=dict(parser=[],answer=identities),
 maximum_parser_requests=0,maximum_embedding_requests=0,maximum_answer_requests=864,maximum_physical_requests=864,stage_cost_cap_usd='8',logical_answers=432,
 prompt_variant_by_identity=variants,answer_instructions_sha256_by_identity={i:[R.P.sha(T.transform_prompt(p,variants[i]))for p in prompts]for i in identities},
 parameters=old_manifest['parameters'],original_U_union=False,profile_retuning=False,configuration_contract_raw_sha256=T.E.file_digest(BASE/'configuration_contract.json'))
R.X.frozen(BASE/'execution_manifest.json',manifest)
R.X.frozen(BASE/'source_inventory.json',T.read(SOURCE69/'source_inventory.json'))
T.write(BASE/'preparation_resources.json',resources70(start,cpu,provider_calls=0,parser_rows_reused=72,query_vectors_reused=24))
print('70 준비 동결: 24질문/72추출·24벡터 재사용/신규답변432/최대864물리. API0.')


70 준비 동결: 24질문/72추출·24벡터 재사용/신규답변432/최대864물리. API0.


## 새 DB SQL 재조회·정확 범위의 원질문 U 검색 재사용

네트워크 호출 없이 검증·조립한다. 원질문/운영 프로필/기존 조건 추출 결과는 고정한다.


In [4]:
start=time.perf_counter();cpu=time.process_time();manifest=T.read(BASE/'execution_manifest.json');T.guard_manifest(manifest)
queries=T.read(BASE/'questions.json')['queries'];vectors=np.load(BASE/'query_vectors.npy',allow_pickle=False)
parsers={(r['query_id'],r['repeat']):r for r in T.read(BASE/'parser_results.json')['records']}
DB5=BASE/'db_v5';DB3=ROOT/'notebooks/data/60_structured_retrieval_repair/db_v3'
facts5=R.R.fact_map(DB5);facts3=R.R.fact_map(DB3)
documents=[json.loads(line)for line in(DB5/'documents.jsonl').read_text().splitlines()]
old_rows=T.read(SOURCE69/'retrieval_results.json')['records'];old_manifest=T.read(SOURCE69/'execution_manifest.json')
reusable=H.validated_prior_searches(old_rows,SOURCE69,old_manifest,manifest,queries,vectors,parsers)
from types import SimpleNamespace
live=[]
def search70_if_needed(*args,**kwargs):
 if not live:live.append(V.Runtime(70))
 return live[0].search(*args,**kwargs)
runtime=SimpleNamespace(path=BASE,manifest=manifest,reused_searches=reusable,search=search70_if_needed)
searches=[];routing_pairs=[]
for index,(q,vector)in enumerate(zip(queries,vectors,strict=True)):
 original=R.S.original_scope(q['question'],documents)
 for repeat in(1,2,3):
  parser=parsers[q['query_id'],repeat]
  route65=T.K.route(R,parser['plan'],original,DB3,facts3,matcher='baseline',parser_failed=parser['status']!='ok')
  route70=T.K.route(R,parser['plan'],original,DB5,facts5,matcher='projected',parser_failed=parser['status']!='ok')
  routes=dict(U=dict(route='U',scope=original['card_keys'],reason='independent_unstructured',db_attempted=False),SEQ65=route65,SEQ_REPAIRED=route70)
  routing_pairs.append(dict(query_id=q['query_id'],repeat=repeat,parser_row_sha256=R.P.sha(parser),original_scope=original,SEQ65=route65,SEQ_REPAIRED=route70))
  for profile in(PROFILES if(index+repeat)%2 else tuple(reversed(PROFILES))):
   shift=(index+repeat)%3
   for mode in MODES[shift:]+MODES[:shift]:
    searches.append(V.retrieve(runtime,q,vector,profile,repeat,routes[mode],original,mode=mode,parser_binding=R.P.sha(parser)))
 print('70 재조회/원질문 U 재사용',index+1,'/24',flush=True)
assert len(searches)==432 and len({r['identity']for r in searches})==432
R.X.frozen(BASE/'retrieval_results.json',dict(records=searches,original_U_union=False))
R.X.frozen(BASE/'routing_comparisons.json',dict(records=routing_pairs,shared_parser_rows=72))
T.write(BASE/'retrieval_resources.json',resources70(start,cpu,logical_searches=432,typed_SQL_reexecuted=288,
 exact_prior_U_reused=sum(r['reused_search_identity']is not None for r in searches),new_operating_search_runtime_loaded=bool(live),
 local_search_resources=dict(live[0].resources())if live else None,
 online_full_retrieval_latency_seconds=None,note='Offline typed reroute and warm exact U reuse; not fresh online search latency'))


70 재조회/원질문 U 재사용 1 /24
70 재조회/원질문 U 재사용 2 /24
70 재조회/원질문 U 재사용 3 /24
70 재조회/원질문 U 재사용 4 /24
70 재조회/원질문 U 재사용 5 /24
70 재조회/원질문 U 재사용 6 /24
70 재조회/원질문 U 재사용 7 /24
70 재조회/원질문 U 재사용 8 /24
70 재조회/원질문 U 재사용 9 /24
70 재조회/원질문 U 재사용 10 /24
70 재조회/원질문 U 재사용 11 /24
70 재조회/원질문 U 재사용 12 /24
70 재조회/원질문 U 재사용 13 /24
70 재조회/원질문 U 재사용 14 /24
70 재조회/원질문 U 재사용 15 /24
70 재조회/원질문 U 재사용 16 /24
70 재조회/원질문 U 재사용 17 /24
70 재조회/원질문 U 재사용 18 /24
70 재조회/원질문 U 재사용 19 /24
70 재조회/원질문 U 재사용 20 /24
70 재조회/원질문 U 재사용 21 /24
70 재조회/원질문 U 재사용 22 /24
70 재조회/원질문 U 재사용 23 /24
70 재조회/원질문 U 재사용 24 /24


## 원U 보존·자동 구조화 사실 추가·입력 재검증·최악 비용 예약

네트워크 호출 없이 검증·조립한다. 원질문/운영 프로필/기존 조건 추출 결과는 고정한다.


In [5]:
start=time.perf_counter();cpu=time.process_time();manifest=T.read(BASE/'execution_manifest.json');T.guard_manifest(manifest)
searches=T.read(BASE/'retrieval_results.json')['records'];facts5=R.R.fact_map(BASE/'db_v5');config=T.read(BASE/'configuration_contract.json')
jobs=[];packages=[]
for row in searches:
 result=copy.deepcopy(row['result']);package=None
 if row['mode']=='SEQ_REPAIRED':
  package=F.assemble(row['question'],row['result'],row['routing'],facts5)
  result=package['results'][config['profiles'][row['profile']]['input_arm']]
  packages.append(dict(identity=row['identity'],query_id=row['query_id'],profile=row['profile'],repeat=row['repeat'],original_U_result_sha256=R.P.sha(row['result']),**{k:v for k,v in package.items()if k!='results'}))
 jobs.append(dict(identity=row['identity'],query_id=row['query_id'],question=row['question'],profile=row['profile'],mode=row['mode'],repeat=row['repeat'],result=result,retrieval_status=row['retrieval_status'],fixed_search_identity=row['identity'],retrieval_row_sha256=R.P.sha(row),package_applicable=package['applicable']if package else False))
assert len(jobs)==432 and len(packages)==144
inventory=H.reconstruct_inventory(jobs,searches,facts5)
R.X.frozen(BASE/'packages.json',dict(records=packages,automatic_only=True,no_U_eviction=True))
R.X.frozen(BASE/'jobs.json',dict(records=jobs))
R.X.frozen(BASE/'additional_fact_inventory.json',dict(status='actual_suffix_facts_pending_full_source_gate',records=inventory,unique_fact_count=len(inventory),manual_gold_in_assembler=False))
R.X.frozen(BASE/'answer_payload_contract.json',A.payload_contract(jobs,manifest))
# DB 경로 값을 다시 비교한 뒤 저장 JSON 순서로 payload를 재구성한다.
lookup={(r['query_id'],r['repeat']):r for r in T.read(BASE/'routing_comparisons.json')['records']}
for job,row in zip(jobs,searches,strict=True):
 q=next(q for q in queries if q['query_id']==row['query_id']);parser=parsers[q['query_id'],row['repeat']]
 original=R.S.original_scope(q['question'],documents);assert original==row['original_scope']
 if row['mode']!='U':
  db,facts,matcher=(BASE/'db_v5',facts5,'projected')if row['mode']=='SEQ_REPAIRED'else(DB3,facts3,'baseline')
  fresh=T.K.route(R,parser['plan'],original,db,facts,matcher=matcher,parser_failed=parser['status']!='ok')
  assert {k:v for k,v in fresh.items()if k!='db_wall_seconds'}=={k:v for k,v in row['routing'].items()if k!='db_wall_seconds'}
 if row['mode']=='SEQ_REPAIRED':
  rebuilt=F.assemble(q['question'],row['result'],row['routing'],facts5)
  assert rebuilt['results'][config['profiles'][row['profile']]['input_arm']]==job['result']
 else:assert row['result']==job['result']
 assert job['result']['evidence'][:len(row['result']['evidence'])]==row['result']['evidence']
R.X.frozen(BASE/'planned_input_verification.json',dict(status='all432_frozen_inputs_verified_before_source_gate',logical_jobs=432,
 typed_SQL_replayed_both_DB_versions=288,original_U_prefix_preserved=True,manual_gold_in_assembler=False,
 reconstructed_additional_inventory_sha256=R.P.sha(inventory),
 files_sha256={str((BASE/n).relative_to(ROOT)):T.E.file_digest(BASE/n)for n in('jobs.json','retrieval_results.json','additional_fact_inventory.json','answer_payload_contract.json')}))
# 입력 byte 최악 비용+출력2400을 논리당 두 물리 시도로 예약한다. 측정 비용과 구분한다.
from pickcardu_rag.answering import serialize_answer_payload,_generated_answer_model
reservation=Decimal(0);bounds=[]
for job in jobs:
 variant=manifest['prompt_variant_by_identity'][job['identity']];evidence=job['result']['evidence']
 schema=_generated_answer_model(len(evidence)).model_json_schema()
 wire=dict(model='gpt-6-luna',instructions=T.transform_prompt(prompts[0],variant),input=[dict(role='user',content=serialize_answer_payload(job['question'],evidence))],schema=schema,tools=[],store=False,reasoning={'effort':'medium'},max_output_tokens=2400)
 bound=max(R.R.request_upper_bound({**wire,'instructions':T.transform_prompt(prompt,variant)},2400)for prompt in prompts)
 reservation+=2*bound;bounds.append(dict(identity=job['identity'],per_physical_upper_usd=str(bound)))
ledger=T.budget_ledger();assert ledger['unknown_cost_records']==0
assert Decimal(ledger['budget_debit_usd'])+reservation<=Decimal('8'),('Worst-case matrix does not fit remaining budget',str(reservation))
R.X.frozen(BASE/'cost_reservation.json',dict(status='full432_times2_attempts_worst_case_reserved_before_answers',maximum_physical=864,
 worst_case_new_usd=str(reservation),starting_debit_usd=ledger['budget_debit_usd'],cumulative_max_usd=str(Decimal(ledger['budget_debit_usd'])+reservation),
 estimate_not_measured_cost=True,records=bounds,maximum_output_tokens=2400,new_unknown_usage_stops=True))
T.write(BASE/'assembly_resources.json',resources70(start,cpu,logical_jobs=432,transmitted_unique_facts=len(inventory),provider_calls=0))
print('70 입력432/실제 추가사실',len(inventory),'최악 추가비용',reservation,'; 원문 gate 대기')


70 입력432/실제 추가사실 55 최악 추가비용 5.271184750 ; 원문 gate 대기


## 원문 gate 통과 뒤 실제432답변 생성

이 셀은 main의 실제 추가 사실 전수 원문 감사와 수락 파일이 없으면 실행할 수 없다. ALLOW_NETWORK 기본값은0이다.


In [10]:
# 사람의 원문 판정은 기존55행 감사와 이번3행/이미지 재검토 기록에서 가져온다.
# 정확히 같은 행만 재사용하며, 새로운 의미를 정답표로 자동 보충하지 않는다.
start=time.perf_counter();cpu=time.process_time();manifest=T.read(BASE/'execution_manifest.json');T.guard_manifest(manifest)
facts5=R.R.fact_map(BASE/'db_v5');jobs=T.read(BASE/'jobs.json')['records'];searches=T.read(BASE/'retrieval_results.json')['records']
actual=H.reconstruct_inventory(jobs,searches,facts5)
assert T.read(BASE/'additional_fact_inventory.json')['records']==actual
old_review=T.read(SOURCE69/'source_review/source_review.json');T.E.verify_file_map(old_review['files_sha256'],ROOT,fresh=True)
old_inventory={r['fact_id']:r for r in T.read(SOURCE69/'additional_fact_inventory.json')['records']}
old_records={r['fact_id']:r for r in old_review['records']}
patch_review=T.read(BASE/'changed_rows_source_review.json');T.E.verify_file_map(patch_review['files_sha256'],ROOT,fresh=True)
patches={r['fact_id']:r for r in patch_review['records']};assert len(patches)==3 and patch_review['confirmed_source_conflicts']==[]
visual_receipt_path=BASE/'visual_field_manual_observations.json';visual_receipt=T.read(visual_receipt_path)
T.E.verify_file_map(visual_receipt['files_sha256'],ROOT,fresh=True)
visual_fields={(o['fact_id'],o['field_path']):o for o in visual_receipt['observations']};assert len(visual_fields)==30
records=[];changed=[];identical=[]
line_shas={json.loads(line)['fact_id']:R.P.sha(line.encode())for line in(BASE/'db_v5/facts.jsonl').read_text().splitlines(keepends=True)}
for entry in actual:
 fid=entry['fact_id'];fact=entry['database_fact'];prior=old_records[fid]
 assert prior['database_fact_sha256']==old_inventory[fid]['database_fact_sha256']
 rec=copy.deepcopy(prior);rec['parent69_record_sha256']=R.P.sha(prior)
 if entry['database_fact_sha256']!=prior['database_fact_sha256']:
  assert fid in patches and patches[fid]['database_fact_sha256']==entry['database_fact_sha256']
  assert fact['fields']['cap']==patches[fid]['new_normalized_cap']
  assert fact['derived']['cap_state']=='unknown'and fact['derived']['cap_krw']is None
  oldfact=old_inventory[fid]['database_fact']
  assert {k:v for k,v in fact.items()if k not in('fields','derived','field_evidence')}=={k:v for k,v in oldfact.items()if k not in('fields','derived','field_evidence')}
  assert {k:v for k,v in fact['fields'].items()if k!='cap'}=={k:v for k,v in oldfact['fields'].items()if k!='cap'}
  assert {k:v for k,v in fact['derived'].items()if not k.startswith('cap')}=={k:v for k,v in oldfact['derived'].items()if not k.startswith('cap')}
  for field in rec['field_reviews']:
   section,name=field['field_path'].split('.',1)
   if (section=='fields'and name=='cap')or(section=='derived'and name.startswith('cap')):
    field['stored_value']=copy.deepcopy(fact[section][name])
    field['status']='source_supported'if section=='fields'else'conservative_state_preserved'
    field['reason']='이번main 원문 재검토: 할인받기 전 이용금액 기준과 자기 항목 금액을 함께 보존했다. 지급한도/기간은unknown이며0원/무제한/계산지급액으로 새 확정하지 않는다.'
  rec.update(source_status='reviewed_source_supported_with_disclosed_omissions',confirmed_conflict_fields=[],
   relationship_ambiguous_fields=[p for p in rec['relationship_ambiguous_fields']if p!='fields.cap'],
   manual_reason='69에서 확인한 cap 의미 충돌을 일반 같은행 provenance 규칙으로 복구했다. 다른 필드·원문·원U는 바꾸지 않았다. 원문 같은 문장의 공과금/디지털구독·멤버십 대상 이용금액 관계를main이 직접 확인했다.',
   source_to_fact_crop_diagnosis=dict(parent69_diagnosis=prior.get('source_to_fact_crop_diagnosis'),current='원문 qualifier를 복구한 normalized cap과 두 출처 span. raw_fact는 원형 보존.'))
  changed.append(fid)
 else:
  assert fact==old_inventory[fid]['database_fact']
  assert prior['confirmed_conflict_fields']==[]
  identical.append(fid)
 rec.update(database_fact_sha256=entry['database_fact_sha256'],database_line_raw_sha256_including_newline=line_shas[fid],database_path=str((BASE/'db_v5/facts.jsonl').relative_to(ROOT)),
  fields=copy.deepcopy(fact['fields']),derived=copy.deepcopy(fact['derived']),field_evidence_sha256=R.P.sha(fact['field_evidence']),
  occurrences=copy.deepcopy(entry['occurrences']),actual70_occurrence_binding=True)
 for field in rec['field_reviews']:
  if not field.get('source_proof_ids'):
   observation=visual_fields[fid,field['field_path']]
   assert observation['database_fact_sha256']==entry['database_fact_sha256']
   field['visual_field_observation']=copy.deepcopy(observation)
 records.append(rec)
assert len(records)==55 and len(changed)==1 and len(identical)==54
files=copy.deepcopy(old_review['files_sha256']);files.update(patch_review['files_sha256']);files.update(visual_receipt['files_sha256'])
for path in [SOURCE69/'source_review/source_review.json',BASE/'changed_rows_source_review.json',visual_receipt_path,
             BASE/'db_changes.json',BASE/'db_migration_receipt.json',BASE/'reaffirmed68_selection.json',BASE/'cost_reservation.json']:
 files[str(path.relative_to(ROOT))]=T.E.file_digest(path)
for name in('execution_manifest.json','jobs.json','retrieval_results.json','additional_fact_inventory.json','answer_payload_contract.json','planned_input_verification.json','configuration_contract.json'):
 files[str((BASE/name).relative_to(ROOT))]=T.E.file_digest(BASE/name)
review=dict(status='all_actual70_added_facts_source_covered_after_cap_context_repair',records=records,
 reviewed_fact_sha256_by_id={r['fact_id']:r['database_fact_sha256']for r in actual},reconstructed_additional_inventory_sha256=R.P.sha(actual),
 source_proof_catalog=copy.deepcopy(old_review['source_proof_catalog']),registered_visual_reviews=copy.deepcopy(old_review['registered_visual_reviews']),
 visual_field_receipt=dict(path=str(visual_receipt_path.relative_to(ROOT)),raw_sha256=T.E.file_digest(visual_receipt_path)),
 files_sha256=files,confirmed_source_conflicts=[],relationship_ambiguous=[dict(fact_id=r['fact_id'],fields=r['relationship_ambiguous_fields'])for r in records if r['relationship_ambiguous_fields']],
 source_judgment_reuse=dict(exact_unchanged_DB_facts=identical,new_cap_context_rechecks=changed,all_changed_DB_rows_reviewed=3,visual_only_fields_rechecked=30),
 coverage=dict(unique_facts=55,occurrences=sum(len(r['occurrences'])for r in actual),field_items=sum(len(r['field_reviews'])for r in records),E55_proofs=len(old_review['source_proof_catalog']),registered_visual_reviews=len(old_review['registered_visual_reviews'])),
 manual_gold_in_assembler=False,semantics='Same-value prior Codex source judgments reused with exact SHA; new cap/visual relations manually reviewed. Code validates coverage/identity/proofs, not independent semantic truth. Known raw-U Monthly conflict and disclosed relation ambiguities remain; production promotion false.')
H.validate_source_records(actual,review,ROOT)
T.write(BASE/'source_review.json',review)
files[str((BASE/'source_review.json').relative_to(ROOT))]=T.E.file_digest(BASE/'source_review.json')
gate=dict(status='accepted_source_and_input_assembly_before_generation',files_sha256=files,
 reviewed_fact_sha256_by_id=review['reviewed_fact_sha256_by_id'],reconstructed_additional_inventory_sha256=R.P.sha(actual),
 confirmed_source_conflicts=0,unreviewed_fact_ids=[],all_transmitted_unique_facts_reviewed=True,manual_gold_in_assembler=False,
 source_review_path=str((BASE/'source_review.json').relative_to(ROOT)),answer_generation_allowed=True,
 cumulative_budget_upper_usd=T.read(BASE/'cost_reservation.json')['cumulative_max_usd'],production_promotion=False,
 posthoc_development_only=True,source_truth_independently_certified=False)
T.write(BASE/'pre_answer_acceptance.json',gate)
# 실제 runtime gate 통과를 확인한다. 이 셀은 API를 호출하지 않는다.
_,verified_jobs,_=H.require_source_gate(BASE,facts5)
assert len(verified_jobs)==432 and not list((BASE/'api_cache').glob('*.json'))
T.write(BASE/'source_gate_runtime_check.json',dict(status='actual70_source_gate_passed_before_any_answer_API',logical_jobs=432,reviewed_facts=55,confirmed_conflicts=0,
 source_review_raw_sha256=T.E.file_digest(BASE/'source_review.json'),acceptance_raw_sha256=T.E.file_digest(BASE/'pre_answer_acceptance.json'),paid70raw_files=0))
T.write(BASE/'source_audit_validation_resources.json',resources70(start,cpu,provider_calls=0,manual_reading_time_seconds=None))
print('70 실제 source gate 통과: 55행/1650필드/54동일원문판정 재사용+1cap수정 재검토/이미지30필드main재검토/확정충돌0/API0.')


70 실제 source gate 통과: 55행/1650필드/54동일원문판정 재사용+1cap수정 재검토/이미지30필드main재검토/확정충돌0/API0.


In [ ]:
if not ALLOW_NETWORK:raise RuntimeError('ALLOW_NETWORK=0: 유료 실행 금지')
manifest,jobs,gate=H.require_source_gate(BASE,R.R.fact_map(BASE/'db_v5'))
ledger=T.budget_ledger();assert ledger['unknown_cost_records']==0
start=time.perf_counter();cpu=time.process_time();rows=[]
expected={j['identity']:R.P.sha(j)for j in jobs};disk_sha=T.E.file_digest(BASE/'jobs.json');gate_sha=T.E.file_digest(BASE/'pre_answer_acceptance.json')
for index,job in enumerate(jobs,1):
 if R.P.sha(job)!=expected[job['identity']]or T.E.file_digest(BASE/'jobs.json')!=disk_sha or T.E.file_digest(BASE/'pre_answer_acceptance.json')!=gate_sha:
  raise RuntimeError('Frozen70 payload or source gate changed')
 rows.append(A.generate(70,job,allow_network=ALLOW_NETWORK))
 if index%12==0:print('70 답변',index,'/432, 누적보수차감',T.budget_ledger()['budget_debit_usd'],flush=True)
R.X.frozen(BASE/'execution_results.json',dict(records=rows,logical_answers=432,fresh_logical_identity_responses=True))
T.write(BASE/'answer_resources.json',resources70(start,cpu,logical_answers=432,physical_events=sum(len(r['events'])for r in rows),
 inclusive_answer_ready_wall_seconds=sum(r['answer_ready_wall_seconds']for r in rows),
 new_response_reuse_across_repeats=False,online_full_pipeline_time_measured=False))
print('70 답변432행 생성완료. 자동 정답채점 없음; 이후 실제 화면 수동 검토.')
